In [1]:
# Cleaning the IEA Global EV Outlook data (European comparison)

In [2]:
# Bring in pandas
import pandas as pd

In [6]:
# Load the IEA data
# 1. Read the Excel file from data/raw
# 2. Pick the main data sheet (GEVO_EV_2026); the file also has notes and summary sheets
# 3. Check the shape: expect (49375, 9)

iea = pd.read_excel("C:/Users/kraus/Desktop/Ironhack/ev-charging-gap/data/raw/EV data by country 2026.xlsx", sheet_name="GEVO_EV_2026")

print(iea.shape)
iea.head()

(49375, 9)


,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
0,Chile,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,1.0,Other
1,Norway,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,18.0,Other
2,Portugal,Historical,EV charging points,EVSE,Publicly available fast,2011,charging points,6.0,Other
3,Canada,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,2.0,Other
4,Chile,Historical,EV charging points,EVSE,Publicly available fast,2012,charging points,4.0,Other


In [7]:
# Keep only measured data for individual countries (plus the EU as a benchmark)
# 1. Keep Historical rows only (drop the projections)
# 2. Keep countries ("Other" and "Projection_country") and the EU total
#    Drop regional totals (World, Europe, Africa...) to avoid double counting
# 3. Check: expect (41648, 9) and 63 places (62 countries + the EU)

keep_groups = ["Other", "Projection_country", "European Union"]

iea_hist = iea[
    (iea["category"] == "Historical") &
    (iea["Aggregate group"].isin(keep_groups))
]

print(iea_hist.shape)
print(iea_hist["region_country"].nunique())

(41648, 9)
63


In [8]:
# Demand side: battery electric cars on the road, per country per year
# 1. Keep rows where the measure is EV stock (cars in use, not new sales)
# 2. ...for Cars only (not vans, buses, trucks or 2-wheelers)
# 3. ...and battery electric only (BEV, not plug-in hybrids), matching the UK analysis
# 4. Keep just three columns: country, year, value, and give them clear names
# 5. Check: expect (633, 3) and 47 places

bev_stock = iea_hist[
    (iea_hist["parameter"] == "EV stock") &
    (iea_hist["mode"] == "Cars") &
    (iea_hist["powertrain"] == "BEV")
]

bev_stock = bev_stock[["region_country", "year", "value"]].rename(columns={
    "region_country": "country",
    "value": "bev_stock"
})

print(bev_stock.shape)
print(bev_stock["country"].nunique())
bev_stock.head()

(633, 3)
47


,country,year,bev_stock
4442,Austria,2010,350.0
4443,Belgium,2010,61.0
4444,Denmark,2010,59.0
4445,Finland,2010,23.0
4446,France,2010,460.0


In [9]:
# Supply side: public charging points, per country per year
# 1. Keep rows where the measure is EV charging points
# 2. Pivot: turn the three speed rows (slow, fast, ultra-fast) into three columns,
#    so each country-year sits on one row (the opposite of melt)
# 3. Turn country and year back into normal columns (reset_index)
# 4. Give the columns clear names
# 5. Check: expect (458, 5) and 44 places

points = iea_hist[iea_hist["parameter"] == "EV charging points"]

points = points.pivot_table(
    index=["region_country", "year"],
    columns="powertrain",
    values="value"
).reset_index()

points = points.rename(columns={
    "region_country": "country",
    "Publicly available slow": "slow_points",
    "Publicly available fast": "fast_points",
    "Publicly available ultra-fast": "ultrafast_points"
})

print(points.shape)
print(points["country"].nunique())
points.head()

(458, 5)
44


powertrain,country,year,fast_points,slow_points,ultrafast_points
0,Australia,2017,40.0,440.0,NaN
1,Australia,2018,61.0,670.0,NaN
2,Australia,2019,260.0,1700.0,NaN
3,Australia,2020,480.0,2000.0,NaN
4,Australia,2021,530.0,2000.0,250.0


In [10]:
# Add up charging points, skipping missing values
# 1. fast_plus_points: fast + ultra-fast (everything over 22kW, the road-trip measure)
# 2. total_points: slow + fast + ultra-fast
# 3. Use .sum(axis=1) so a missing value is skipped rather than wiping out the total
#    (min_count=1: if ALL the values are missing, the result stays missing, not 0)
# 4. Check: Australia 2017 = 40 fast-plus, 480 total; UK 2025 = 27000 fast-plus, 116000 total

points["fast_plus_points"] = points[["fast_points", "ultrafast_points"]].sum(axis=1, min_count=1)

points["total_points"] = points[["slow_points", "fast_points", "ultrafast_points"]].sum(axis=1, min_count=1)

points.head()

powertrain,country,year,fast_points,slow_points,ultrafast_points,fast_plus_points,total_points
0,Australia,2017,40.0,440.0,NaN,40.0,480.0
1,Australia,2018,61.0,670.0,NaN,61.0,731.0
2,Australia,2019,260.0,1700.0,NaN,260.0,1960.0
3,Australia,2020,480.0,2000.0,NaN,480.0,2480.0
4,Australia,2021,530.0,2000.0,250.0,780.0,2780.0


In [11]:
# Check the UK in 2025: expect 27000 fast-plus and 116000 total

points[(points["country"] == "United Kingdom") & (points["year"] == 2025)]

powertrain,country,year,fast_points,slow_points,ultrafast_points,fast_plus_points,total_points
448,United Kingdom,2025,15000.0,89000.0,12000.0,27000.0,116000.0


In [12]:
# Put cars and charging points side by side (the pandas version of a SQL JOIN)
# 1. Match rows on the same country AND the same year
# 2. Inner join: keep only country-years that appear in BOTH tables
#    (drops countries with car data but no charging data, e.g. Cambodia)
# 3. Check: expect (457, 8) and 44 places (43 countries + the EU)

europe_compare = bev_stock.merge(points, on=["country", "year"], how="inner")

print(europe_compare.shape)
print(europe_compare["country"].nunique())
europe_compare.head()

(457, 8)
44


,country,year,bev_stock,fast_points,slow_points,ultrafast_points,fast_plus_points,total_points
0,Netherlands,2010,270.0,NaN,400.0,NaN,NaN,400.0
1,Norway,2010,2700.0,NaN,2800.0,NaN,NaN,2800.0
2,Chile,2011,6.0,1.0,1.0,NaN,1.0,2.0
3,Netherlands,2011,1100.0,NaN,400.0,NaN,NaN,400.0
4,Norway,2011,3900.0,18.0,3100.0,NaN,18.0,3118.0


In [14]:
# Calculate the ratios, mark European countries, and save
# 1. Battery electric cars per public charging point (any speed)
# 2. Battery electric cars per fast charging point (over 22kW, the road-trip measure)
# 3. Mark the 18 European countries with complete data (is_europe)
#    and the EU total as a benchmark (is_eu_benchmark), so Tableau can filter them
# 4. Save to data/clean

europe_compare["bevs_per_point"] = europe_compare["bev_stock"] / europe_compare["total_points"]
europe_compare["bevs_per_fast_point"] = europe_compare["bev_stock"] / europe_compare["fast_plus_points"]

europe = [
    "Austria", "Belgium", "Denmark", "Finland", "France", "Germany",
    "Greece", "Iceland", "Italy", "Netherlands", "Norway", "Poland",
    "Portugal", "Spain", "Sweden", "Switzerland", "Turkiye", "United Kingdom"
]

europe_compare["is_europe"] = europe_compare["country"].isin(europe)
europe_compare["is_eu_benchmark"] = europe_compare["country"] == "European Union"

europe_compare.to_csv("../data/clean/europe_comparison.csv", index=False)

In [15]:
# Check: the 2025 European comparison, sorted by cars per fast charger
# Expect 18 countries + the EU; the UK at 13.8 per point and 59.3 per fast point

check = europe_compare[
    (europe_compare["year"] == 2025) &
    (europe_compare["is_europe"] | europe_compare["is_eu_benchmark"])
]

check.sort_values("bevs_per_fast_point").round(1)

,country,year,bev_stock,fast_points,slow_points,ultrafast_points,fast_plus_points,total_points,bevs_per_point,bevs_per_fast_point,is_europe,is_eu_benchmark
347,Greece,2025,24000.0,1200.0,7100.0,340.0,1540.0,8640.0,2.8,15.6,True,False
364,Spain,2025,310000.0,11000.0,37000.0,5600.0,16600.0,53600.0,5.8,18.7,True,False
351,Italy,2025,340000.0,10000.0,54000.0,4500.0,14500.0,68500.0,5.0,23.4,True,False
368,Turkiye,2025,360000.0,8800.0,21000.0,6300.0,15100.0,36100.0,10.0,23.8,True,False
336,Austria,2025,260000.0,5800.0,26000.0,4100.0,9900.0,35900.0,7.2,26.3,True,False
359,Poland,2025,120000.0,2800.0,7400.0,1600.0,4400.0,11800.0,10.2,27.3,True,False
344,Finland,2025,160000.0,1100.0,15000.0,4000.0,5100.0,20100.0,8.0,31.4,True,False
456,European Union,2025,7300000.0,120000.0,890000.0,110000.0,230000.0,1120000.0,6.5,31.7,False,True
360,Portugal,2025,180000.0,4100.0,9900.0,1200.0,5300.0,15200.0,11.8,34.0,True,False
345,France,2025,1400000.0,18000.0,150000.0,22000.0,40000.0,190000.0,7.4,35.0,True,False
